# Faruq-v3 — AF2 vs CLAHE parallel control

Jalankan notebook ini pada **3 runtime Colab terpisah**. Pada masing-masing runtime set `SEED` menjadi `42`, `123`, atau `2026`. Ketiga runtime menulis ke subfolder seed yang berbeda di Drive. Protokol ilmiah tetap frozen: D0FT vs CLAHE_LAB vs AF2, validation-only, test tidak boleh tersedia.


In [ ]:
# ===== UBAH HANYA INI DI MASING-MASING RUNTIME =====
SEED = 42  # runtime lain: 123 dan 2026
assert SEED in (42, 123, 2026)
print('WORKER SEED:', SEED)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/af2-clahe-control'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO, check=True)
else:
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for key in list(sys.modules):
    if key == 'coffee_detector' or key.startswith('coffee_detector.'):
        sys.modules.pop(key, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())


In [ ]:
import cv2, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan GPU pada runtime ini.'
REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-af2-igem-paired-confirmation-v1/val_reports/af2_igem_paired_confirmation.json',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed2026/weights/best.pt',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
AF2_CONFIRMATION = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
D0_BY_SEED = {
    42: require_project_artifact(PROJECT_ROOT, REQUIRED[2]),
    123: require_project_artifact(PROJECT_ROOT, REQUIRED[3]),
    2026: require_project_artifact(PROJECT_ROOT, REQUIRED[4]),
}
D0_CHECKPOINT = D0_BY_SEED[SEED]
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-af2-vs-clahe-control-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / 'CLAHE_LAB' / f'CLAHE_LAB_seed{SEED}'
print('GPU       :', torch.cuda.get_device_name(0))
print('OpenCV    :', cv2.__version__)
print('SEED      :', SEED)
print('D0        :', D0_CHECKPOINT)
print('RUN DIR   :', RUN_DIR)
print('OUTPUTROOT:', OUTPUT_ROOT)


In [ ]:
# Train/resume hanya seed milik runtime ini. Output seed lain tidak disentuh.
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_af2_clahe_worker',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--af2-confirmation', str(AF2_CONFIRMATION),
    '--d0-checkpoint', str(D0_CHECKPOINT),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', str(SEED),
    '--device', '0', '--authorize-training',
]
RUN_LOG = Path(f'/content/af2_vs_clahe_seed{SEED}.log')
print('MENJALANKAN:', ' '.join(command), flush=True)
with RUN_LOG.open('a', encoding='utf-8', buffering=1) as log_stream:
    process = subprocess.Popen(command, cwd=REPO, text=True, stdout=log_stream, stderr=subprocess.STDOUT)
    while process.poll() is None:
        csv_path = RUN_DIR / 'results.csv'
        epochs = 0
        if csv_path.is_file():
            try:
                import pandas as pd
                epochs = len(pd.read_csv(csv_path))
            except Exception:
                pass
        print(f'[SEED {SEED}] {epochs}/50 epoch', flush=True)
        time.sleep(60)
    return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(RUN_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-160:])
    print(tail)
    raise RuntimeError(f'Worker seed {SEED} gagal; log={RUN_LOG}')
print('WORKER SELESAI:', SEED)


In [ ]:
# Tampilkan hasil seed ini.
import pandas as pd
from IPython.display import display
WORKER = OUTPUT_ROOT / 'val_reports' / f'CLAHE_LAB_seed{SEED}_worker.json'
assert WORKER.is_file(), WORKER
worker = json.loads(WORKER.read_text(encoding='utf-8'))
rows = [{'model': model, **worker[model]} for model in ('D0FT', 'CLAHE_LAB', 'AF2')]
display(pd.DataFrame(rows).style.format({
    'macro_map50_95': '{:.2%}',
    'bottom3_class_map50_95': '{:.2%}',
    'worst_class_map50_95': '{:.2%}',
}))
print('WORKER SUMMARY:', WORKER)


In [ ]:
# OPTIONAL: jalankan ini dari salah satu runtime setelah SEMUA 3 seed selesai.
report_dir = OUTPUT_ROOT / 'val_reports'
ready = all((report_dir / f'CLAHE_LAB_seed{s}_worker.json').is_file() for s in (42,123,2026))
print('SEMUA SEED READY:', ready)
if ready:
    collect_cmd = [
        sys.executable, '-m', 'coffee_detector.experiments.collect_faruq_v3_af2_clahe_parallel',
        '--af2-confirmation', str(AF2_CONFIRMATION),
        '--output-root', str(OUTPUT_ROOT),
    ]
    subprocess.run(collect_cmd, cwd=REPO, check=True)
    SUMMARY = report_dir / 'af2_vs_clahe_classical_enhancement_control.json'
    result = json.loads(SUMMARY.read_text(encoding='utf-8'))
    rows = []
    for seed, values in result['per_seed'].items():
        for model in ('D0FT', 'CLAHE_LAB', 'AF2'):
            rows.append({'seed': seed, 'model': model, **values[model]})
    display(pd.DataFrame(rows).style.format({
        'macro_map50_95': '{:.2%}',
        'bottom3_class_map50_95': '{:.2%}',
        'worst_class_map50_95': '{:.2%}',
    }))
    print(json.dumps(result['decisions'], indent=2))
    print('INTERPRETATION:', result['interpretation'])
    print('SUMMARY:', SUMMARY)
else:
    print('Biarkan dua runtime lain menyelesaikan seed mereka, lalu rerun cell ini.')
